# LangSmith Prompts in Code

## What are Prompts in LangSmith?

**Prompts** are the instructions you give to Large Language Models (LLMs) to guide their behavior. LangSmith provides a centralized way to:
- **Store** prompts as versioned templates
- **Manage** different versions with commit history
- **Collaborate** with your team on prompt improvements
- **Reuse** prompts across different applications

## Why Use LangSmith for Prompt Management?

1. **Version Control**: Track changes to your prompts over time
2. **Collaboration**: Multiple team members can iterate on prompts
3. **Centralized Storage**: One source of truth for all your prompts
4. **Environment Management**: Use tags like `prod`, `dev`, `staging`
5. **Dynamic Templates**: Create reusable prompts with variables

---

## Prerequisites

Before starting, make sure you have:
- A LangSmith account (sign up at https://smith.langchain.com)
- Your LangSmith API key set as an environment variable
- Required Python packages installed


In [ ]:
import os
from dotenv import load_dotenv
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()

# Verify that required environment variables are set
if not os.getenv("LANGSMITH_API_KEY"):
    print("⚠️  Warning: LANGSMITH_API_KEY not found!")
    print("   Make sure you have a .env file with LANGSMITH_API_KEY set")
    print("   Or set it manually: os.environ['LANGSMITH_API_KEY'] = 'your-key'")

if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Warning: OPENAI_API_KEY not found!")
    print("   Make sure you have a .env file with OPENAI_API_KEY set")

# Initialize LangSmith client
client = Client()

print("✅ Setup complete! LangSmith client initialized.")
print(f"📍 Project: {os.getenv('LANGSMITH_PROJECT', 'default')}")


---

## 1. Creating Your First Prompt

Let's start by creating a simple prompt template with variables. We'll use `ChatPromptTemplate` which is the recommended format for chat-based models.

### Key Concepts:
- **System Message**: Sets the behavior/persona of the AI
- **User Message**: Contains the user's input, often with variables like `{question}`
- **Variables**: Placeholders that get filled in at runtime (e.g., `{topic}`, `{question}`)


In [ ]:
# Example 1: Simple chatbot prompt
simple_prompt = ChatPromptTemplate([
    ("system", "You are a helpful and friendly AI assistant."),
    ("user", "{question}"),
])

# Push this prompt to LangSmith
client.push_prompt("my-first-prompt", object=simple_prompt)

print("✅ Prompt 'my-first-prompt' created!")
print("📝 You can view it in LangSmith UI under Prompts section")


In [ ]:
# Example 2: More complex prompt with multiple variables
expert_prompt = ChatPromptTemplate([
    ("system", "You are an expert in {domain}. Provide detailed, accurate answers."),
    ("user", "Question: {question}\n\nContext: {context}"),
])

# Push to LangSmith
client.push_prompt("expert-assistant", object=expert_prompt)

print("✅ Prompt 'expert-assistant' created!")
print("📝 This prompt has 3 variables: domain, question, and context")


---

## 2. Pulling and Using Prompts

Once you've created a prompt in LangSmith, you can pull it from anywhere in your code. This is useful for:
- **Reusing prompts** across different applications
- **Separating prompt logic** from your code
- **Allowing non-developers** to update prompts without changing code


In [ ]:
# Pull a prompt from LangSmith
pulled_prompt = client.pull_prompt("my-first-prompt")

print("✅ Prompt pulled successfully!")
print(f"\nPrompt structure:")
print(pulled_prompt)

# You can also inspect the messages
print(f"\nPrompt messages:")
for message in pulled_prompt.messages:
    print(f"  - {message}")


In [ ]:
# Format the prompt with actual values
formatted_messages = pulled_prompt.format_messages(question="What is the capital of France?")

print("📨 Formatted messages:")
for msg in formatted_messages:
    print(f"\n{msg.type.upper()}:")
    print(f"  {msg.content}")


In [ ]:
complex_pulled_prompt = client.pull_prompt("expert-assistant")

complex_formatted_messages = complex_pulled_prompt.format_messages(domain="AI", question="Why would you use LangGraph?", context="LangGraph is a framework for building production-grade LLM applications.")

print(" Formatted messages:")
for msg in complex_formatted_messages:
    print(f"\n{msg.type.upper()}:")
    print(f"  {msg.content}")


---

## 3. Using Prompts with LLMs

Now let's combine our prompts with actual language models. LangSmith supports storing prompts **with** model configurations.


In [ ]:
# Create a chain: prompt + model
model = ChatOpenAI(model="gpt-4o", temperature=0.7)

# Option 1: Create chain manually
prompt_from_ls = client.pull_prompt("my-first-prompt")
chain = prompt_from_ls | model

# Use the chain
response = chain.invoke({"question": "What are the benefits of using LangSmith for prompt management?"})
print("🤖 AI Response:")
print(response.content)


### Storing Prompts with Model Configurations

You can store the prompt **and** model configuration together. This is useful when you want to version both the prompt and the specific model settings.


In [ ]:
# Create a joke generator prompt
joke_prompt = ChatPromptTemplate([
    ("system", "You are a witty comedian who tells funny, clean jokes."),
    ("user", "Tell me a joke about {topic}"),
])

# Create a chain with specific model configuration
joke_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)  # Higher temperature for creativity
joke_chain = joke_prompt | joke_model

# Push the ENTIRE chain (prompt + model) to LangSmith
client.push_prompt("joke-generator-with-model", object=joke_chain)

print("✅ Prompt with model configuration saved!")
print("📝 This includes both the prompt template AND the model settings")


In [ ]:
# Pull the chain (prompt + model) from LangSmith
# Use include_model=True to get both prompt and model
joke_chain_from_ls = client.pull_prompt("joke-generator-with-model", include_model=True)

# Use it directly - no need to attach a model!
response = joke_chain_from_ls.invoke({"topic": "programming"})
print("😄 Joke:")
print(response.content)


---

## 4. Prompt Versioning and Commit Tags

Every time you push a prompt to LangSmith, it creates a new **commit** (version). This is similar to Git commits!

### Why Use Commit Tags?

Commit tags let you:
- Mark specific versions as `production`, `staging`, or `dev`
- Reference stable versions without hardcoding commit IDs
- Easily roll back to previous versions
- Switch between versions without changing code

### Best Practices:

1. **Use tags for environments**: Tag prompts with `prod`, `dev`, `staging`
2. **Version your releases**: Use tags like `v1.0`, `v1.1`, `v2.0`
3. **Test before promoting**: Test in `dev` before tagging as `prod`


In [ ]:
# Update an existing prompt
updated_prompt = ChatPromptTemplate([
    ("system", "You are a helpful and friendly AI assistant. Always be concise and clear."),
    ("user", "{question}"),
])

# Push the updated version
client.push_prompt("my-first-prompt", object=updated_prompt)

print("✅ Prompt updated!")
print("📝 A new commit has been created in LangSmith")
print("💡 You can tag this commit in the UI as 'v1.1' or 'production'")


### Pulling Specific Versions

You can pull prompts by:
- **Latest version** (default): `client.pull_prompt("prompt-name")`
- **Specific tag**: `client.pull_prompt("prompt-name:tag-name")`  
- **Specific commit**: `client.pull_prompt("prompt-name", version="commit-hash")`

> **Note**: Tags must be created in the LangSmith UI. Navigate to your prompt → Commits tab → Add Commit Tag


In [ ]:
# Example: Pull the latest version
latest_prompt = client.pull_prompt("my-first-prompt")
print("✅ Pulled latest version of prompt")

# Example: Pull a specific tagged version (after creating tag in UI)
production_prompt = client.pull_prompt("my-first-prompt:production")
dev_prompt = client.pull_prompt("my-first-prompt:dev")

# Display prompts in a readable format
print("\n📝 Production Prompt:")
for msg in production_prompt.messages:
    if hasattr(msg, 'prompt'):
        print(f"  {msg.__class__.__name__.replace('MessagePromptTemplate', '')}: {msg.prompt.template}")

print("\n📝 Dev Prompt:")
for msg in dev_prompt.messages:
    if hasattr(msg, 'prompt'):
        print(f"  {msg.__class__.__name__.replace('MessagePromptTemplate', '')}: {msg.prompt.template}")

# This allows you to switch between environments easily!
print("\n💡 Tip: Use tags to reference different versions in different environments:")
print("   - Development: pull_prompt('my-prompt:dev')")
print("   - Staging: pull_prompt('my-prompt:staging')")
print("   - Production: pull_prompt('my-prompt:production')")


---

## 5. Using Public Prompts from LangChain Hub

LangSmith has a **public prompt hub** where the community shares useful prompts. You can browse, fork, and use these prompts!

### How to Use Public Prompts:

1. Browse prompts at: **LangSmith UI → Prompts → Browse Public Prompts**
2. Pull public prompts using: `{author-handle}/{prompt-name}`

> **Note**: Public prompts are user-generated and unverified. Review them before use!


In [ ]:
# Example: Pull a public prompt from LangChain Hub
# The langchain-ai organization has many useful prompts

# Note: This will only work if you have access to the hub
# You can also use the legacy langchain.hub (from langchain import hub)
try:
    # Pull a public evaluation prompt as an example
    public_prompt = client.pull_prompt("langchain-ai/retrieval-qa-chat")
    print("✅ Successfully pulled public prompt from LangChain Hub")
    print(f"\nPrompt structure:")
    print(public_prompt)
except Exception as e:
    print(f"⚠️ Could not pull public prompt: {e}")
    print("💡 You can browse public prompts in the LangSmith UI")


---

## 6. Advanced Patterns and Best Practices

### Pattern 1: Multi-Turn Conversations

Create prompts that support chat history for conversational applications.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Create a conversational prompt with history
conversational_prompt = ChatPromptTemplate([
    ("system", "You are a helpful assistant. Use the conversation history to provide context-aware responses."),
    ("placeholder", "{chat_history}"),  # This will be filled with past messages
    ("user", "{question}"),
])

# Push to LangSmith
client.push_prompt("conversational-assistant", object=conversational_prompt)

# Simulate a conversation
chat_history = [
    HumanMessage(content="My name is Alice"),
    AIMessage(content="Nice to meet you, Alice! How can I help you today?"),
]

# Create chain and invoke with history
conv_chain = conversational_prompt | model
response = conv_chain.invoke({
    "chat_history": chat_history,
    "question": "What's my name?"
})

print("🤖 AI Response (with memory):")
print(response.content)


### Pattern 2: Few-Shot Prompting

Few-shot prompting provides examples to guide the model's behavior.


In [ ]:
# Create a few-shot classification prompt
few_shot_prompt = ChatPromptTemplate([
    ("system", """You are a sentiment classifier. Classify the sentiment as positive, negative, or neutral.

Examples:
Input: "I love this product!"
Output: positive

Input: "This is terrible."
Output: negative

Input: "It's okay, nothing special."
Output: neutral"""),
    ("user", "Input: {text}\nOutput:"),
])

# Push to LangSmith
client.push_prompt("sentiment-classifier", object=few_shot_prompt)

# Test it
sentiment_chain = few_shot_prompt | model
result = sentiment_chain.invoke({"text": "This is absolutely amazing!"})

print("🎯 Classification Result:")
print(result.content)


### Pattern 3: RAG (Retrieval-Augmented Generation) Prompts

RAG prompts incorporate retrieved context to answer questions.


In [ ]:
# Create a RAG prompt
rag_prompt = ChatPromptTemplate([
    ("system", """You are a helpful assistant that answers questions based on the provided context.
    
Instructions:
- Use ONLY the information from the context to answer
- If the answer is not in the context, say "I don't have enough information to answer that."
- Be concise and accurate"""),
    ("user", """Context: {context}

Question: {question}

Answer:"""),
])

# Push to LangSmith
client.push_prompt("rag-qa-assistant", object=rag_prompt)

# Test it with sample context
context = """
LangSmith is a platform for building production-grade LLM applications.
It provides tools for debugging, testing, evaluating, and monitoring.
LangSmith supports prompt versioning and team collaboration.
"""

rag_chain = rag_prompt | model
result = rag_chain.invoke({
    "context": context,
    "question": "What is LangSmith?"
})

print("📚 RAG Response:")
print(result.content)


### Pattern 4: Template Formats - F-String vs Mustache

LangSmith supports two template formats for variables:
- **F-string** (default): Uses `{variable}` - Simple and familiar to Python developers
- **Mustache**: Uses `{{variable}}` - More powerful with conditionals, loops, and nested keys

> 💡 **When to use Mustache**: Use mustache when you need conditional logic, loops, or complex variable handling

Reference: [LangSmith Prompt Engineering Concepts](https://docs.langchain.com/langsmith/prompt-engineering-concepts)

Mustache Reference: [Mustache Manual](https://mustache.github.io/mustache.5.html)

In [ ]:
# Example 1: F-string format (default)
fstring_prompt = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("user", "Hello, {name}! How can I help you today?"),
])

client.push_prompt("fstring-example", object=fstring_prompt)

# Test it
result = fstring_prompt.format_messages(name="Alice")
print("📝 F-string format output:")
for msg in result:
    print(f"{msg.type}: {msg.content}")

print("\n✅ F-string is simple and works great for basic variable substitution!")


#### Mustache Template Syntax Explained

##### The Three Key Symbols

**1. `#` (Hash) - "If True" / "Loop"**
```mustache
{{#variable}}
  This content shows when variable is truthy (true, non-empty, exists)
{{/variable}}
```

**2. `/` (Slash) - "Closing Tag"**
```mustache
{{#variable}}
  Content here
{{/variable}}  ← This closes the block started with #
```

**3. `^` (Caret) - "If False" / "Negation"**
```mustache
{{^variable}}
  This content shows when variable is falsy (false, empty, null)
{{/variable}}
```

##### Key Rules

1. **Every `#` needs a matching `/`** - They work like opening and closing tags
2. **`#variable`** = "Show this if variable is truthy OR loop if it's an array"
3. **`^variable`** = "Show this if variable is falsy" (opposite of `#`)
4. **Always close with `/variable`** - Whether you used `#` or `^` to open

---

##### Visual Pattern

Think of it like HTML tags:

```
#  = Opening tag  (like <if>)
/  = Closing tag  (like </if>)
^  = Negation tag (like <if-not>)
```

```mustache
{{#condition}}        ← If TRUE
  Content A
{{/condition}}        ← Close

{{^condition}}        ← If FALSE  
  Content B
{{/condition}}        ← Close
```

---

##### Quick Cheat Sheet

| Syntax | Meaning | Use Case |
|--------|---------|----------|
| `{{variable}}` | Insert value | Simple substitution |
| `{{#variable}}...{{/variable}}` | If true / Loop | Show content if truthy, or iterate array |
| `{{^variable}}...{{/variable}}` | If false | Show content if falsy |
| `{{/variable}}` | Close block | Required to close `#` or `^` |

In [ ]:
# Example 2: Mustache format with conditionals
# Note: To use mustache in the LangSmith Playground, switch the template format in settings

from langchain_core.prompts import PromptTemplate

# Mustache prompt with conditional logic
mustache_prompt_text = """{{#is_logged_in}}
Welcome back, {{name}}! Here are your recent activities.
{{/is_logged_in}}{{^is_logged_in}}
Please log in to continue, {{name}}.
{{/is_logged_in}}"""

# Create a PromptTemplate with mustache-style variables
mustache_prompt = PromptTemplate.from_template(
    mustache_prompt_text,
    template_format="mustache"
)

print("📝 Mustache format examples:")
print("\n1. When user IS logged in:")
result1 = mustache_prompt.format(is_logged_in=True, name="Alice")
print(result1)

print("\n2. When user is NOT logged in:")
result2 = mustache_prompt.format(is_logged_in=False, name="Bob")
print(result2)

print("\n✅ Mustache supports conditionals with {{#variable}} and {{^variable}} (negation)!")


In [ ]:
# Example 3: Real-world mustache example - Personalized customer support
customer_support_mustache = """You are a customer support agent.

Customer Information:
- Name: {{customer_name}}
- Account Type: {{account_type}}
{{#is_premium}}
- Premium Features: Enabled
- Priority Support: Yes
{{/is_premium}}
{{^is_premium}}
- Premium Features: Not available (upgrade to access)
- Support: Standard queue
{{/is_premium}}

{{#has_open_ticket}}
IMPORTANT: This customer has an open ticket #{{ticket_id}}.
Reference this ticket in your response.
{{/has_open_ticket}}

Respond professionally to: {{question}}"""

mustache_support = PromptTemplate.from_template(
    customer_support_mustache,
    template_format="mustache"
)

# Push to LangSmith
# Note: When viewing in the UI, switch to mustache format in prompt settings
client.push_prompt("mustache-customer-support", object=mustache_support)

# Test with different scenarios
print("🎯 Scenario 1: Premium customer with open ticket")
scenario1 = mustache_support.format(
    customer_name="Alice Johnson",
    account_type="Enterprise",
    is_premium=True,
    has_open_ticket=True,
    ticket_id="12345",
    question="I need help with the integration"
)
print(scenario1)

print("\n" + "="*60)
print("🎯 Scenario 2: Standard customer, no ticket")
scenario2 = mustache_support.format(
    customer_name="Bob Smith",
    account_type="Basic",
    is_premium=False,
    has_open_ticket=False,
    question="How do I upgrade my account?"
)
print(scenario2)

print("\n✅ Mustache conditionals enable dynamic, context-aware prompts!")


#### Mustache Template Features

Mustache provides several powerful features beyond simple variable substitution:

**1. Conditionals:**
- `{{#variable}}...{{/variable}}` - Shows content if variable is truthy
- `{{^variable}}...{{/variable}}` - Shows content if variable is falsy (negation)
- `{{#variable}}...{{else}}...{{/variable}}` - If-else logic

**2. Loops:**
```
{{#items}}
  - {{name}}: {{price}}
{{/items}}
```

**3. Nested Keys:**
```
{{user.profile.name}}
{{user.settings.theme}}
```

**Using Mustache in LangSmith Playground:**
1. Open your prompt in the Playground
2. Click the gear icon (⚙️) next to the model name
3. Go to "Template Format" section
4. Select "mustache" instead of "f-string"
5. For conditional variables, manually add JSON in the 'inputs' section

**Example JSON for conditionals:**
```json
{
  "name": "Alice",
  "is_logged_in": true,
  "has_open_ticket": false
}
```

> 📚 Learn more: [Mustache Documentation](https://mustache.github.io/mustache.5.html)


In [ ]:
# Example 4: Mustache with loops - Product recommendation
product_recommendation = """You are a helpful shopping assistant.

Customer: {{customer_name}}

Available Products:
{{#products}}
- {{name}}: ${{price}}
  {{#on_sale}}⭐ ON SALE! {{discount}}% off{{/on_sale}}
{{/products}}

Based on these products and the customer's interest in {{interest}}, provide a personalized recommendation."""

mustache_shopping = PromptTemplate.from_template(
    product_recommendation,
    template_format="mustache"
)

# Test with product data
products_data = [
    {"name": "Laptop Pro", "price": "1299", "on_sale": True, "discount": 15},
    {"name": "Wireless Mouse", "price": "49", "on_sale": False},
    {"name": "USB-C Hub", "price": "79", "on_sale": True, "discount": 20},
]

result = mustache_shopping.format(
    customer_name="Sarah",
    products=products_data,
    interest="productivity tools"
)

print("🛍️ Product Recommendation Prompt:")
print(result)
print("\n✅ Mustache loops make it easy to handle dynamic lists of data!")


#### When to Use F-String vs Mustache

| Feature | F-String `{var}` | Mustache `{{var}}` |
|---------|------------------|-------------------|
| **Simple variables** | ✅ Perfect | ✅ Works well |
| **Conditionals** | ❌ Not supported | ✅ Full support |
| **Loops** | ❌ Not supported | ✅ Full support |
| **Nested keys** | ⚠️ Limited | ✅ Full support |
| **Ease of use** | ⭐⭐⭐⭐⭐ Very simple | ⭐⭐⭐ More complex |
| **Python familiarity** | ✅ Native Python | ⚠️ New syntax |

**Choose F-String when:**
- You only need simple variable substitution
- Your team is familiar with Python
- You don't need conditional logic

**Choose Mustache when:**
- You need conditional content based on variables
- You're working with dynamic lists/arrays
- You need complex nested data structures
- Non-developers will be editing prompts in the UI


---

## 7. Best Practices for LangSmith Prompts

### 🎯 Naming Conventions

- Use descriptive, kebab-case names: `customer-support-chatbot`
- Include purpose in name: `sentiment-classifier`, `code-reviewer`
- Avoid generic names like `prompt1`, `test-prompt`

### 📋 Prompt Design Tips

1. **Be Specific**: Clearly define the task and expected output format
2. **Use System Messages**: Set the AI's role and behavior
3. **Include Examples**: Few-shot examples improve consistency
4. **Handle Edge Cases**: Tell the model what to do when uncertain
5. **Test Thoroughly**: Use the Playground to test with various inputs

### 🔄 Version Management

- **Tag Important Versions**: Mark stable versions with tags
- **Use Environment Tags**: Separate `dev`, `staging`, `prod`
- **Document Changes**: Add clear commit messages in the UI
- **Test Before Promoting**: Always test in lower environments first

### 🤝 Team Collaboration

- **Share Prompts**: Use LangSmith as the single source of truth
- **Iterate in Playground**: Test changes before pushing
- **Review Together**: Use the UI to compare versions
- **Set Permissions**: Control who can edit production prompts

### ⚡ Performance Tips

- **Be Concise**: Shorter prompts are faster and cheaper
- **Optimize Variables**: Only include necessary context
- **Cache When Possible**: Reuse prompts across requests
- **Monitor Usage**: Track prompt performance in LangSmith


---

## 8. Complete Example: Building a Production-Ready Prompt

Let's put it all together with a real-world example: a customer support chatbot.


In [ ]:
# Step 1: Create a production-ready prompt
customer_support_prompt = ChatPromptTemplate([
    ("system", """You are a professional customer support agent for TechCorp, a software company.

Your responsibilities:
- Be friendly, empathetic, and professional
- Provide accurate information from the knowledge base
- If you don't know something, offer to escalate to a human agent
- Keep responses concise (under 150 words)

Company policies:
- Refunds available within 30 days
- 24/7 technical support available
- Enterprise customers get priority support"""),
    ("placeholder", "{chat_history}"),
    ("user", """Customer: {customer_name}
Customer Type: {customer_type}
Question: {question}"""),
])

# Step 2: Combine with model configuration
support_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,  # Balanced creativity
    max_tokens=200    # Keep responses concise
)

support_chain = customer_support_prompt | support_model

# Step 3: Push to LangSmith with descriptive name
client.push_prompt("customer-support-agent-v1", object=support_chain)

print("✅ Production-ready customer support prompt created!")
print("📝 Next steps:")
print("   1. Test in LangSmith Playground")
print("   2. Run evaluations on test dataset")
print("   3. Tag as 'staging' for testing")
print("   4. After validation, tag as 'production'")


In [ ]:
# Step 4: Use the prompt in your application
def handle_customer_query(customer_name, customer_type, question, history=None):
    """
    Production function that uses the LangSmith prompt
    """
    # Pull the production version
    support_chain = client.pull_prompt("customer-support-agent-v1:production", include_model=True)
    
    # Invoke with customer data
    response = support_chain.invoke({
        "customer_name": customer_name,
        "customer_type": customer_type,
        "question": question,
        "chat_history": history or []
    })
    
    return response.content

# Test the function
# Note: This will fail if 'production' tag doesn't exist yet
# You can remove ':production' to use the latest version
try:
    result = handle_customer_query(
        customer_name="John Smith",
        customer_type="Enterprise",
        question="How do I reset my password?"
    )
    print("💬 Support Agent Response:")
    print(result)
except Exception as e:
    print(f"⚠️ Could not use production tag: {e}")
    print("💡 Tag your prompt as 'production' in the UI, or use the latest version instead")


---

## 9. Using Prompts Without LangChain (Optional)

LangSmith is framework-agnostic! You can use prompts with OpenAI or Anthropic SDKs directly.


In [ ]:
import openai

# Pull prompt from LangSmith
prompt = client.pull_prompt("my-first-prompt")

# Convert to OpenAI format
# LangSmith provides conversion methods for popular APIs
messages = prompt.format_messages(question="What is machine learning?")

# Convert LangChain messages to OpenAI format
openai_messages = [
    {"role": msg.type, "content": msg.content} 
    for msg in messages
]

print("🔄 Converted to OpenAI format:")
print(openai_messages)
print("\n💡 You can now use this with the OpenAI SDK directly!")

# Example (commented out to avoid API calls):
# openai_client = openai.OpenAI()
# response = openai_client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=openai_messages
# )
# print(response.choices[0].message.content)


---

## 10. Key Takeaways

### What You Learned 🎓

1. ✅ **Create prompts** using `client.push_prompt()`
2. ✅ **Pull prompts** using `client.pull_prompt()`
3. ✅ **Use variables** to make prompts dynamic and reusable
4. ✅ **Version prompts** with commits and tags
5. ✅ **Store model configs** alongside prompts
6. ✅ **Use environment tags** for deployment workflows
7. ✅ **Apply patterns** like RAG, few-shot, and conversations
8. ✅ **Follow best practices** for production use

### Quick Reference 📚

```python
# Create and push
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

client = Client()
prompt = ChatPromptTemplate([("system", "..."), ("user", "{var}")])
client.push_prompt("prompt-name", object=prompt)

# Pull and use
prompt = client.pull_prompt("prompt-name")
prompt = client.pull_prompt("prompt-name:production")  # With tag
chain = client.pull_prompt("name", include_model=True)  # With model

# Use
result = chain.invoke({"var": "value"})
```

### Next Steps 🚀

1. **Explore the LangSmith UI**
   - Navigate to the Prompts section
   - Try the Playground to test prompts interactively
   - Compare different versions side-by-side

2. **Run Evaluations**
   - Create datasets in LangSmith
   - Use the Playground to run experiments
   - Track performance over time

3. **Set Up Production Workflow**
   - Use `dev` → `staging` → `production` tags
   - Integrate with your CI/CD pipeline
   - Set up webhooks for prompt updates

4. **Collaborate with Your Team**
   - Share prompts with team members
   - Review and iterate together
   - Document your prompt strategies

### Additional Resources 📖

- **LangSmith Documentation**: https://docs.langchain.com/langsmith
- **LangSmith Playground**: https://smith.langchain.com
- **Public Prompt Hub**: Browse community prompts in the UI
- **LangSmith API Reference**: https://docs.langchain.com/langsmith/smith-python-sdk

---

## 🎉 Congratulations!

You're now ready to use LangSmith for professional prompt management. Start building, iterating, and shipping better prompts faster!
